# 🧪 Lab 03: Follow the Plan

Welcome to the plan-transformation autopsy bay. In this lab, we keep one query unchanged and change only the evidence we ask Spark to show us.

**Mission Objective:** distinguish the logical plan, the physical plan, WholeStageCodegen stages, and generated Java. The notebook is deliberately not a benchmark. Its job is to prevent several different meanings of “compile” from collapsing into one vague Spark story.

**Deterministic Guardrail:** one native DataFrame query, no UDF, no external data, and no configuration changes between the different plan views.


### Step 1: Define the Spark diagnostic session
We disable adaptive execution so the physical plan and generated-stage evidence remain directly comparable.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
    .master("local[2]")
    .appName("lab-03-follow-the-plan")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("WholeStageCodegen enabled:", spark.conf.get("spark.sql.codegen.wholeStage"))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 06:22:28 WARN Utils: Your hostname, T14-PF4WM3XL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 06:22:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/08/24 06:22:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0
WholeStageCodegen enabled: true


### Step 2: Write the query once
The query contains a filter, a native projection, and an aggregation. We preview the result only as a sanity check; the later cells inspect the original query unchanged.


In [2]:
query = (spark.range(0, 1_000_000)
    .where((F.col("id") % 7) == 0)
    .select((F.col("id") * 3 + 1).alias("x"))
    .groupBy((F.col("x") % 10).alias("bucket"))
    .agg(F.sum("x").alias("total")))

print("Result preview:")
query.show(5, truncate=False)


Result preview:


+------+-----------+
|bucket|total      |
+------+-----------+
|2     |21428242842|
|4     |21428842854|
|5     |21429142860|
|8     |21430042878|
|1     |21427942836|
+------+-----------+
only showing top 5 rows


### Step 3: Observe logical analysis and optimization
The extended explanation shows parsed and analyzed logical plans, followed by the optimized logical plan. This is Catalyst evidence: what the query means and how Spark rewrote it before selecting concrete execution operators.


In [3]:
print("=== Extended plan: logical analysis and optimization ===")
query.explain("extended")


=== Extended plan: logical analysis and optimization ===
== Parsed Logical Plan ==
'Aggregate ['`%`('x, 10) AS bucket#5], ['`%`('x, 10) AS bucket#5, 'sum('x) AS total#6]
+- Project [((id#0L * cast(3 as bigint)) + cast(1 as bigint)) AS x#4L]
   +- Filter ((id#0L % cast(7 as bigint)) = cast(0 as bigint))
      +- Range (0, 1000000, step=1, splits=Some(2))

== Analyzed Logical Plan ==
bucket: bigint, total: bigint
Aggregate [(x#4L % cast(10 as bigint))], [(x#4L % cast(10 as bigint)) AS bucket#5L, sum(x#4L) AS total#6L]
+- Project [((id#0L * cast(3 as bigint)) + cast(1 as bigint)) AS x#4L]
   +- Filter ((id#0L % cast(7 as bigint)) = cast(0 as bigint))
      +- Range (0, 1000000, step=1, splits=Some(2))

== Optimized Logical Plan ==
Aggregate [_groupingexpression#20L], [_groupingexpression#20L AS bucket#5L, sum(x#4L) AS total#6L]
+- Project [x#4L, (x#4L % 10) AS _groupingexpression#20L]
   +- Project [((id#0L * 3) + 1) AS x#4L]
      +- Filter ((id#0L % 7) = 0)
         +- Range (0, 1000000

### Step 4: Observe the selected physical operators
The formatted view exposes concrete operators such as `Range`, `Filter`, `Project`, `HashAggregate`, and `Exchange`. This is the execution shape that WholeStageCodegen examines for compatible fragments.


In [4]:
print("=== Formatted physical plan ===")
query.explain("formatted")


=== Formatted physical plan ===
== Physical Plan ==
* HashAggregate (7)
+- Exchange (6)
   +- * HashAggregate (5)
      +- * Project (4)
         +- * Project (3)
            +- * Filter (2)
               +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 1000000, step=1, splits=Some(2))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : ((id#0L % 7) = 0)

(3) Project [codegen id : 1]
Output [1]: [((id#0L * 3) + 1) AS x#4L]
Input [1]: [id#0L]

(4) Project [codegen id : 1]
Output [2]: [x#4L, (x#4L % 10) AS _groupingexpression#20L]
Input [1]: [x#4L]

(5) HashAggregate [codegen id : 1]
Input [2]: [x#4L, _groupingexpression#20L]
Keys [1]: [_groupingexpression#20L]
Functions [1]: [partial_sum(x#4L)]
Aggregate Attributes [1]: [sum#12L]
Results [2]: [_groupingexpression#20L, sum#13L]

(6) Exchange
Input [2]: [_groupingexpression#20L, sum#13L]
Arguments: hashpartitioning(_groupingexpression#20L, 2), ENSURE_REQUIREMENTS, [plan_id=71]

(7) HashAggregat

### Step 5: Observe the generated Java
The codegen explanation is a different kind of evidence. It exposes generated iterators and Java source fragments for eligible physical regions. It is not the logical plan, and it is not the JVM JIT’s final machine code.


In [5]:
print("=== Generated code for eligible stages ===")
query.explain("codegen")


=== Generated code for eligible stages ===


Found 2 WholeStageCodegen subtrees.
== Subtree 1 / 2 (maxMethodCodeSize:374; maxConstantPoolSize:382(0.58% used); numInnerClasses:2) ==
*(1) HashAggregate(keys=[_groupingexpression#20L], functions=[partial_sum(x#4L)], output=[_groupingexpression#20L, sum#13L])
+- *(1) Project [x#4L, (x#4L % 10) AS _groupingexpression#20L]
   +- *(1) Project [((id#0L * 3) + 1) AS x#4L]
      +- *(1) Filter ((id#0L % 7) = 0)
         +- *(1) Range (0, 1000000, step=1, splits=2)

Generated code:
/* 001 */ public Object generate(Object[] references) {
/* 002 */   return new GeneratedIteratorForCodegenStage1(references);
/* 003 */ }
/* 004 */
/* 005 */ // codegenStageId=1
/* 006 */ final class GeneratedIteratorForCodegenStage1 extends org.apache.spark.sql.execution.BufferedRowIterator {
/* 007 */   private Object[] references;
/* 008 */   private scala.collection.Iterator[] inputs;
/* 009 */   private boolean hashAgg_initAgg_0;
/* 010 */   private boolean hashAgg_bufIsNull_0;
/* 011 */   private long hashAg

# 📊 Post-Lab Analysis: The Plan Becomes a Program

This lab held the query constant and changed only the inspection layer. The evidence separates three stages of the story: Catalyst describes and rewrites the query, the physical planner chooses executable operators, and WholeStageCodegen emits Java for compatible physical fragments.

### 1. Logical Plans Describe Meaning

The extended plan shows parsed, analyzed, and optimized logical plans. This is where Spark resolves columns and types and applies logical rewrites. Catalyst is not yet showing the final generated Java loop.

### 2. Physical Plans Choose Machinery

The formatted plan shows concrete execution operators and boundaries. `Exchange` is a physical data-movement operator; the `*(n)` markers and codegen IDs identify the compatible regions that can be generated together.

### 3. Generated Java Is a Separate Layer

The codegen view exposes generated iterators and Java source for the selected stages. Spark code generation is not the same thing as Catalyst optimization, runtime Java compilation, or the JVM JIT turning bytecode into machine code.

The same query therefore produces three different kinds of testimony: logical plan means **what Spark understands**, physical plan means **how Spark intends to execute it**, and generated code means **what Spark writes for eligible fragments**. The plan was written first. Only then did Spark begin writing the program.
